In [ ]:
import numpy as np
from scipy import stats
from scipy.optimize import minimize

from zne import fold_circuit
from circuits import generate_ising_benchmark

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import Estimator as AerEstimator
from qiskit_aer.noise import NoiseModel, depolarizing_error

In [ ]:
def simulate_noisy_distribution(expval_ideal, lambda_scale, mu, sigma, shots_per_batch, size):
    N = np.random.poisson(lambda_scale, size=size)
    total_errors = np.sum(N)
    
    if total_errors > 0:
        error_impacts = np.random.normal(mu, sigma, size=total_errors)
        circuit_indices = np.repeat(np.arange(size), N)
        Y = np.bincount(circuit_indices, weights=error_impacts, minlength=size)
    else:
        Y = np.zeros(size)
    
    simulated_values = expval_ideal - Y
    
    shot_noise = np.random.normal(0, 1.0 / np.sqrt(shots_per_batch), size=size)
    simulated_values += shot_noise
    
    return simulated_values

def global_ks_loss(params, data_dict, shots_per_batch, sim_size=10000):
    expval_ideal, lambda_base, mu, sigma = params
    
    if lambda_base < 0 or sigma < 0 or abs(expval_ideal) > 1.2:
        return np.inf
        
    total_ks_distance = 0.0
    
    for lambda_base, dist in data_dict.items():
        lambda_rate = lambda_base * lambda_scale
        
        proposed_dist = simulate_noisy_distribution(
            expval_ideal, lambda_rate, mu, sigma, shots_per_batch, sim_size
        )
        
        ks_stat, _ = stats.ks_2samp(dist, proposed_dist)
        total_ks_distance += ks_stat
        
    return total_ks_distance

In [ ]:
noise_model = NoiseModel()
error_1q = depolarizing_error(1E-3, 1)
noise_model.add_all_qubit_quantum_error(error_1q, ["x", "y", "z", "h", "s", "t"])
error_2q = depolarizing_error(1E-2, 2)
noise_model.add_all_qubit_quantum_error(error_2q, ["cx"])

qc, obs = generate_ising_benchmark(n_qubits=4, trotter_steps=2)

lambda_scales = [2*k+1 for k in range(10)]
batches = 1
shots_per_batch = int(1E5)

estimator = AerEstimator(
    backend_options={"noise_model": noise_model},
    run_options={"shots": shots_per_batch}
)

print("--- Step 1: Gathering Data ---")
data = {}

for lambda_scale in lambda_scales:
    qc_scaled = fold_circuit(qc, lambda_scale)
    
    result = estimator.run([qc_scaled] * batches, [obs] * batches, optimization_level=0).result()
    values = result.values
    data[lambda_scale] = values
    
    print(f"Scale λ={lambda_scale}: Mean = {np.mean(values):.4f}, Variance = {np.var(values):.4f}")

print("\n--- Step 2: Optimizing Compound Poisson Model ---")
guess_expval_ideal = data[1].mean() + 0.2
guess_base_rate = 1.0
guess_mu = 0.1
guess_sigma = 0.05

initial_guess = [guess_expval_ideal, guess_base_rate, guess_mu, guess_sigma]

result = minimize(
    global_ks_loss,
    initial_guess,
    args=(data, shots_per_batch),
    method="Nelder-Mead",
    options={"maxiter": 1000, "xatol": 1e-3, "fatol": 1e-3}
)

if result.success:
    expval_ideal, lambda_base, mu, sigma = result.x
    
    print("\n--- Fit Results ---")
    print(f"Extrapolated E(ideal): {expval_ideal:.4f}")
    print(f"Base Error Rate (λ=1): {lambda_base:.4f} errors/circuit")
    print(f"Avg Error Impact (μ):  {mu:.4f} amplitude loss per error")
    print(f"Error Impact Var (σ):  {sigma:.4f}")
    print(f"Final Global K-S Loss: {result.fun:.4f}")
else:
    print("Optimizer failed to converge. Try adjusting initial guesses or increasing batch size.")